# Reliability and Validation of the MACE Machine-Learned Interatomic Potential for UiO-66 (N₂ model)

**Model:** `MACE_UiO_N2_multi_stagetwo.model` (Multi-Atomic Cluster Expansion, MACE)
**Reference data:** Density Functional Theory (DFT) single points in `msd_N02_train.extxyz` and `msd_N02_test.extxyz`

Structural characterization of this Metal-Organic Framework (MOF) — Powder X-Ray Diffraction (PXRD), Fourier-Transform Infrared (FTIR) spectroscopy, Pore Size Distribution (PSD), Brunauer–Emmett–Teller (BET) surface area, Radial Distribution Function (RDF) — is already covered in `MOF_Characterization.ipynb`. This notebook adds the four tests that are specific to Machine-Learned Interatomic Potentials (MLIPs), each probing a different order of the potential energy surface (PES):

| # | Test | What it probes | Why it matters for an MLIP |
|---|------|----------------|----------------------------|
| 1 | Held-out energy/force accuracy + novelty analysis | PES values and first derivatives on DFT data | Baseline fitting quality; checks over-fitting and whether the test set is really "new" |
| 2 | Phonon dispersion, density of states (DOS), dynamical stability | Second derivatives (curvature) of the PES | Forces can be accurate while curvature is wrong; imaginary modes expose unphysical minima |
| 3 | Energy conservation in microcanonical (NVE) Molecular Dynamics (MD) | Smoothness / conservativeness of the PES along trajectories | Required before trusting production MD (Mean Squared Displacement (MSD), diffusion) |
| 3B | LAMMPS production-scale NVE, 1×1×1 vs 2×2×2 | Same as Test 3, on the real production engine/model file, at two system sizes | Confirms Test 3's ASE/`.model` result transfers to LAMMPS/`-lammps.pt` and is converged w.r.t. supercell size |
| 4 | Equation of State (EOS) and bulk modulus | Response to volume change — outside the training distribution | Tests extrapolation and the (never trained) stress |

These four follow what recent MOF–MLIP benchmarks use: MOFSimBench (npj Comput. Mater. 2025) evaluates structure optimization, MD stability, bulk modulus from EOS fits and phonon-derived heat capacity; the MACE-MP-MOF0 work (npj Comput. Mater. 2025) validates phonons of UiO-66 and others with Phonopy finite displacements and uses the removal of imaginary modes as a validation metric.

### Facts about the data set (checked directly from the files)
- Train file: 730 frames = 514 `N02` (UiO-66 primitive cell + 2 N₂, 118 atoms) + 36 `repulsion` (rigid N₂ at 1.098 Å pushed toward the framework, N–framework ≥ 2.27 Å) + 180 `gas_N2` (isolated N₂ in a 15 Å box).
- Test file: 81 frames = 57 `N02` + 4 `repulsion` + 20 `gas_N2`.
- Framework cell: FCC primitive (rhombohedral) cell, |a| = 14.770 Å, i.e. conventional cubic a = 20.888 Å, V = 2278.6 Å³. **All MOF frames share this one cell.**
- Composition of the framework part: Zr₆O₃₂C₄₈H₂₈ (114 atoms) = one Zr₆O₄(OH)₄(BDC)₆ unit.
- **No stress/virial labels** exist in the files, and the training command had no `--stress_key`/`--stress_weight`. The volume response of this model was therefore never supervised (important for Test 4).
- Training used `--valid_fraction=0.02`, so ~15 of the 730 "train" frames were validation frames not used for gradient updates; they cannot be separated after the fact, so the "train" errors below are for train + validation together.
- `forces_weight=100`, `energy_weight=10`: the loss is force-dominated, so relative energy errors are expected to be larger than force errors.

### How to run
Run in the `mace` conda environment (mace-torch, the Atomic Simulation Environment (ASE) ≥ 3.23, phonopy ≥ 2.20, spglib, scipy, matplotlib) on a GPU node. Put the `.model` and the two `.extxyz` files next to the notebook. Use the **`.model`** file here; the `-lammps.pt` file is only for LAMMPS `pair_style mace`.

> **Provenance:** all numbers and figures below are real, executed results (not pre-filled). Test 1 (held-out accuracy) runs on MPS/float32 (Apple GPU, sufficient precision for statistical RMSE/MAE over hundreds of frames). Test 2 (relaxation + phonons) and Test 4 (EOS) run on CPU/float64 (`calc_fp64`, set up right after the main calculator) because symmetry detection and small-strain energetics are precision-sensitive; both were re-verified in float64 after an initial float32/MPS pass gave identical qualitative results. The manuscript output is reduced to two main figures (`Fig_MACE_accuracy`, `Fig_MACE_phonon_dispersion`), one summary table (Table 1), and four SI figures (SI1-novelty, SI2-dispersion-convergence, SI3-EOS, SI4-NVE-drift); all are written to `manuscript_figures/` as matched PNG+PDF pairs (plus a CSV/LaTeX pair for Table 1). Everything else computed along the way is still printed/plotted inline for debugging but is not part of the manuscript figure set.

Estimated cost on one V100 in float64: Test 1 ≈ minutes; Test 2 (2×2×2 supercell, ±displacements) ≈ 0.5–2 h; Test 3 ≈ 30–60 min; Test 4 ≈ 10–30 min. Set `QUICK = True` for a fast smoke test first.

**Apple Silicon (M-series) note:** plain CPU inference of this model, and even MPS inference without the thread-count env vars set below, hung indefinitely (unkillable) in this environment -- a known PyTorch/OpenMP deadlock on conda builds for Apple Silicon. The setup cells below auto-select the MPS (Apple GPU) backend instead when no CUDA GPU is present. Metal has no float64 support, so MPS runs are done in float32 (the model is loaded at its native float64 on CPU then cast+moved to MPS); expect first-call warmup of a few seconds and per-frame cost around 150–200 ms on an M3 once warmed up. `torch.linalg.det` (used for the cell volume in the stress calculation) is not implemented on MPS and falls back to CPU automatically via `PYTORCH_ENABLE_MPS_FALLBACK=1`, set below.

---
## 0. Setup — imports, paths, calculator

In [ ]:
import os, time, json, warnings
# Apple Silicon: multiple BLAS/OpenMP thread pools (numpy/scipy/torch) contend for the GIL/thread
# affinity once torch also has to dispatch to the Metal (MPS) queue, and the combination hangs
# indefinitely (an unkillable process, not just slow) -- confirmed on this machine, this is the fix.
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
          "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # let unsupported MPS ops (e.g. linalg.det) fall back to CPU
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import ase
from ase import Atoms, units
from ase.io import read
from ase.optimize import BFGS
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.md.verlet import VelocityVerlet
from ase.md.langevin import Langevin
from ase.spacegroup.symmetrize import refine_symmetry, check_symmetry
try:
    from ase.constraints import FixSymmetry
except ImportError:                                   # moved in some ASE versions
    from ase.spacegroup.symmetrize import FixSymmetry

import scipy
from scipy.optimize import curve_fit

import phonopy
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms

import mace
from mace.calculators import MACECalculator

print(f"torch {torch.__version__} | ase {ase.__version__} | phonopy {phonopy.__version__} | "
      f"mace {getattr(mace, '__version__', '?')} | scipy {scipy.__version__} | numpy {np.__version__}")

In [ ]:
# ----------------------------- user settings -----------------------------
MODEL_PATH = "MACE_UiO_N2_multi_stagetwo.model"   # .model for ASE (NOT the -lammps.pt file)
TRAIN_FILE = "msd_N02_train.extxyz"
TEST_FILE  = "msd_N02_test.extxyz"
OUT_DIR    = "mlip_validation_outputs"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"           # Apple Silicon GPU
else:
    DEVICE = "cpu"
# model was trained with --default_dtype=float64, but Metal/MPS has no float64 support at all
# (Apple GPUs do not implement double precision) -> fall back to float32 on MPS only.
DTYPE  = "float64" if DEVICE != "mps" else "float32"

# Hyper-parameters from the training command (used for receptive-field reasoning)
R_MAX = 5.0                 # Å
N_INTERACTIONS = 2
RECEPTIVE_FIELD = R_MAX * N_INTERACTIONS      # radius of each site-energy environment (Å)

QUICK = False               # True -> short runs for a smoke test
SEED  = 1268
rng   = np.random.default_rng(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.facecolor": "white",
                     "font.size": 11, "axes.linewidth": 0.8})

def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name), bbox_inches="tight", facecolor="white")

FIG_DIR = "manuscript_figures"   # the finalized, publication-ready figures (PNG+PDF)
os.makedirs(FIG_DIR, exist_ok=True)

def savefig_dual(fig, name):
    """Manuscript-figure style: dpi=200, white facecolor, PNG + PDF."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=200, bbox_inches="tight", facecolor="white")

THZ_TO_CM1 = 33.35641        # 1 THz = 33.35641 cm^-1
SUMMARY = {}                 # collects every headline number for the final table
print("device:", DEVICE)

In [ ]:
# model was saved on a CUDA machine; its embedded JIT submodule hardcodes device='cuda',
# so map_location alone does not work on this CPU-only machine
_real_jit_load = torch.jit.load
torch.jit.load = lambda f, map_location=None, **kw: _real_jit_load(f, map_location='cpu', **kw)

# MPS can only materialize float32 tensors directly (this checkpoint's storages are float64,
# and torch.load(..., map_location="mps") fails before default_dtype ever gets applied), so
# always load+cast on CPU first, then move the already-float32 model to the real device.
calc = MACECalculator(model_paths=MODEL_PATH, device="cpu", default_dtype=DTYPE)
if DEVICE != "cpu":
    calc.models = [m.to(DEVICE) for m in calc.models]
    calc.device = torch.device(DEVICE)

def get_reference(atoms):
    """DFT energy/forces stored in the extxyz (ASE attaches them as a SinglePointCalculator)."""
    if atoms.calc is not None and "energy" in atoms.calc.results:
        E = atoms.calc.results["energy"]
        F = atoms.calc.results.get("forces", None)
    else:
        E = atoms.info["energy"]
        F = None
    if F is None:
        F = atoms.arrays["forces"]
    return float(E), np.array(F, dtype=float)

train = read(TRAIN_FILE, index=":")
test  = read(TEST_FILE,  index=":")

for name, frames in [("train", train), ("test", test)]:
    counts = defaultdict(int)
    for a in frames:
        counts[a.info.get("config_type", "default")] += 1
    print(f"{name}: {len(frames)} frames  {dict(counts)}")

mof_train = [a for a in train if a.info.get("config_type") in ("N02", "repulsion")]
vols = np.array([a.get_volume() for a in mof_train])
print(f"MOF-cell volume range in training data: {vols.min():.3f} – {vols.max():.3f} Å^3 "
      f"(spread {100*(vols.max()-vols.min())/vols.mean():.4f} %)")
has_stress = any(("stress" in a.info) or ("virial" in a.info) or
                 (a.calc is not None and "stress" in a.calc.results) for a in train)
print("stress/virial labels present in training data:", has_stress)
SUMMARY["train_volume_A3"] = float(vols.mean())

---
## Test 1 — Held-out accuracy, generalization gap and novelty of the test set

**Metrics** (all computed per config type and per element):
- Energy: Mean Absolute Error (MAE), Root Mean Square Error (RMSE) and mean signed error (bias), in meV/atom.
- Forces: component-wise MAE/RMSE in meV/Å; vector error |ΔF|; cosine similarity between predicted and DFT force vectors.
- Generalization gap: test / train error ratio. A ratio ≫ 1 indicates over-fitting.
- Error vs. DFT force magnitude: shows whether large-force (high-energy, e.g. `repulsion`) configurations are handled.

**Novelty of the test set.** The `N02` frames come from DFT-MD trajectories, and a random train/test split puts temporally adjacent (nearly identical) frames on both sides. A low test error then only demonstrates interpolation. To make this visible, for every test frame we compute the minimum root-mean-square displacement (RMSD, minimum-image convention) of the framework atoms to *any* training frame, and plot the test error against it. Tests 2–4 are needed precisely because Test 1 cannot establish extrapolation.

In [ ]:
def evaluate(frames, calc, label=""):
    rec, t0 = [], time.time()
    for i, at in enumerate(frames):
        E_ref, F_ref = get_reference(at)
        a = at.copy()
        a.calc = calc
        E_pred = a.get_potential_energy()
        F_pred = a.get_forces()
        rec.append(dict(idx=i, config_type=at.info.get("config_type", "default"), n=len(a),
                        symbols=np.array(a.get_chemical_symbols()),
                        E_ref=E_ref, E_pred=float(E_pred), F_ref=F_ref, F_pred=np.array(F_pred)))
    print(f"{label}: {len(frames)} frames evaluated in {time.time()-t0:.1f} s")
    return rec

rec_train = evaluate(train, calc, "train")
rec_test  = evaluate(test,  calc, "test")

In [ ]:
def energy_errors(rec, config_types=None):
    r = [x for x in rec if config_types is None or x["config_type"] in config_types]
    dE = np.array([(x["E_pred"] - x["E_ref"]) / x["n"] for x in r]) * 1000.0   # meV/atom
    return dE

def force_errors(rec, config_types=None, element=None):
    ref, pred, mag = [], [], []
    for x in rec:
        if config_types is not None and x["config_type"] not in config_types:
            continue
        m = np.ones(x["n"], bool) if element is None else (x["symbols"] == element)
        ref.append(x["F_ref"][m]); pred.append(x["F_pred"][m])
    if not ref:
        return None
    ref, pred = np.concatenate(ref), np.concatenate(pred)
    return ref, pred

def stats(v):
    return dict(MAE=float(np.mean(np.abs(v))), RMSE=float(np.sqrt(np.mean(v**2))), bias=float(np.mean(v)),
                max=float(np.max(np.abs(v))), N=int(v.size))

groups = {"all": None, "N02": ["N02"], "repulsion": ["repulsion"], "gas_N2": ["gas_N2"]}
elements = ["Zr", "O", "C", "H", "N"]
table = {}

print(f"{'set':6s} {'group':10s} | {'E MAE':>8s} {'E RMSE':>8s} {'E bias':>8s} (meV/atom) | "
      f"{'F MAE':>7s} {'F RMSE':>7s} (meV/Å) | cos-sim")
for set_name, rec in [("train", rec_train), ("test", rec_test)]:
    for g, ct in groups.items():
        dE = energy_errors(rec, ct)
        fr = force_errors(rec, ct)
        if dE.size == 0 or fr is None:
            continue
        ref, pred = fr
        dF = (pred - ref).ravel() * 1000.0
        nr, npd = np.linalg.norm(ref, axis=1), np.linalg.norm(pred, axis=1)
        ok = (nr > 1e-3) & (npd > 1e-3)
        cos = np.sum(ref[ok] * pred[ok], axis=1) / (nr[ok] * npd[ok])
        sE, sF = stats(dE), stats(dF)
        table[(set_name, g)] = dict(E=sE, F=sF, cos_mean=float(cos.mean()))
        print(f"{set_name:6s} {g:10s} | {sE['MAE']:8.3f} {sE['RMSE']:8.3f} {sE['bias']:8.3f}            | "
              f"{sF['MAE']:7.2f} {sF['RMSE']:7.2f}          | {cos.mean():.4f}")

print("\nPer-element force errors (all config types, meV/Å):")
print(f"{'el':3s} | {'train MAE':>9s} {'train RMSE':>10s} | {'test MAE':>8s} {'test RMSE':>9s} | test/train RMSE")
for el in elements:
    out = []
    for rec in (rec_train, rec_test):
        ref, pred = force_errors(rec, None, el)
        out.append(stats((pred - ref).ravel() * 1000.0))
    ratio = out[1]["RMSE"] / out[0]["RMSE"]
    table[("elem", el)] = dict(train=out[0], test=out[1])
    print(f"{el:3s} | {out[0]['MAE']:9.2f} {out[0]['RMSE']:10.2f} | {out[1]['MAE']:8.2f} {out[1]['RMSE']:9.2f} | {ratio:.2f}")

gapE = table[("test", "all")]["E"]["RMSE"] / table[("train", "all")]["E"]["RMSE"]
gapF = table[("test", "all")]["F"]["RMSE"] / table[("train", "all")]["F"]["RMSE"]
print(f"\nGeneralization gap (test RMSE / train RMSE): energy {gapE:.2f}, forces {gapF:.2f}")

SUMMARY.update({
    "T1_test_E_RMSE_meV_atom": table[("test", "all")]["E"]["RMSE"],
    "T1_test_E_MAE_meV_atom":  table[("test", "all")]["E"]["MAE"],
    "T1_test_F_RMSE_meV_A":    table[("test", "all")]["F"]["RMSE"],
    "T1_test_F_MAE_meV_A":     table[("test", "all")]["F"]["MAE"],
    "T1_gap_E": gapE, "T1_gap_F": gapF,
})

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Figure 1 — MACE vs DFT accuracy (energy + force parity on the held-out test set)
# ══════════════════════════════════════════════════════════════════════════
mof_ct = ["N02", "repulsion"]
Eref_all = np.array([x["E_ref"] / x["n"] for x in rec_train + rec_test if x["config_type"] in mof_ct])
E_shift = Eref_all.mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# (a) energy parity, MOF frames only (N02 + repulsion as one combined test set)
ax = axes[0]
er_tr = np.array([x["E_ref"] / x["n"] for x in rec_train if x["config_type"] in mof_ct]) - E_shift
ep_tr = np.array([x["E_pred"] / x["n"] for x in rec_train if x["config_type"] in mof_ct]) - E_shift
ax.scatter(er_tr * 1000, ep_tr * 1000, s=12, c="tab:red",marker="o", edgecolors="k", linewidths=0, alpha=0.6, label="train", rasterized=True)

r_test_mof = [x for x in rec_test if x["config_type"] in mof_ct]
n_cell = r_test_mof[0]["n"]
er = np.array([x["E_ref"] / x["n"] for x in r_test_mof]) - E_shift
ep = np.array([x["E_pred"] / x["n"] for x in r_test_mof]) - E_shift
ax.scatter(er * 1000, ep * 1000, s=14, c="tab:blue", marker="o",
           edgecolors="k", linewidths=0.3, label="test", zorder=5)

lim = np.array(ax.get_xlim() + ax.get_ylim()); lo, hi = lim.min(), lim.max()
ax.plot([lo, hi], [lo, hi], "k--", lw=0.9, zorder=1)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("DFT energy − mean (meV/atom)", fontweight="bold")
ax.set_ylabel("MACE energy − mean (meV/atom)", fontweight="bold")
ax.legend(fontsize=7.5, frameon=False, loc="lower right")

dE_mof = energy_errors(rec_test, mof_ct)
E_stats = stats(dE_mof)
E_rmse_cell = E_stats["RMSE"] * n_cell
E_mae_cell = E_stats["MAE"] * n_cell
# ax.text(0.03, 0.97, f"test RMSE = {E_stats['RMSE']:.2f} meV/atom\n"
#                      f"             = {E_rmse_cell:.1f} meV/cell\n"
#                      f"test MAE  = {E_stats['MAE']:.2f} meV/atom\n"
#                      f"             = {E_mae_cell:.1f} meV/cell",
#         transform=ax.transAxes, ha="left", va="top", fontsize=7,
#         bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.92))
ax.text(-0.05, 1.1, "(a)", transform=ax.transAxes, fontsize=16, fontweight="bold", va="top")

# (b) force parity, ALL test config types, hexbin
ax = axes[1]
Fref_all, Fpred_all = force_errors(rec_test)
hb = ax.hexbin(Fref_all.ravel(), Fpred_all.ravel(), gridsize=150, cmap="viridis", bins="log", mincnt=1)
lo, hi = min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([lo, hi], [lo, hi], "k--", lw=0.9, zorder=5)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("DFT force component (eV/Å)", fontweight="bold")
ax.set_ylabel("MACE force component (eV/Å)", fontweight="bold")
cb = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("count")
tab_txt = "el   RMSE(meV/Å)\n" + "\n".join(f"{el:<3s} {table[('elem', el)]['test']['RMSE']:6.1f}" for el in elements)
#ax.text(0.03, 0.97, tab_txt, transform=ax.transAxes, ha="left", va="top", fontsize=6.5,
#        family="monospace", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.92))
ax.text(-0.05, 1.1, "(b)", transform=ax.transAxes, fontsize=16, fontweight="bold", va="top")

fig.tight_layout()
savefig_dual(fig, "Fig_MACE_accuracy")
plt.show()
print("Saved Fig_MACE_accuracy.png/.pdf ->", FIG_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SI-1 — Novelty of the test set vs. training data
# ══════════════════════════════════════════════════════════════════════════
def framework_mask(atoms):
    return np.array([s != "N" for s in atoms.get_chemical_symbols()])

mof_test  = [(i, a) for i, a in enumerate(test) if a.info.get("config_type") == "N02"]
mof_trn   = [a for a in train if a.info.get("config_type") in ("N02", "repulsion")]

ref_syms = mof_trn[0].get_chemical_symbols()
same_order = all(a.get_chemical_symbols() == ref_syms for a in mof_trn) and \
             all(a.get_chemical_symbols() == ref_syms for _, a in mof_test)
same_cell = np.allclose(np.array([a.cell[:] for a in mof_trn]), mof_trn[0].cell[:], atol=1e-4)
print("identical atom ordering:", same_order, "| identical cell:", same_cell)

if same_order and same_cell:
    cell = mof_trn[0].cell[:]
    fm = framework_mask(mof_trn[0])
    S_trn = np.array([a.get_scaled_positions(wrap=False)[fm] for a in mof_trn])
    d_min = []
    for i, a in mof_test:
        s = a.get_scaled_positions(wrap=False)[fm]
        ds = S_trn - s[None]
        ds -= np.round(ds)
        dr = ds @ cell
        rmsd = np.sqrt(np.mean(np.sum(dr**2, axis=2), axis=1))
        d_min.append(rmsd.min())
    d_min = np.array(d_min)
    rec_by_idx = {x["idx"]: x for x in rec_test}
    f_rmse = np.array([np.sqrt(np.mean((rec_by_idx[i]["F_pred"] - rec_by_idx[i]["F_ref"])**2)) * 1000
                       for i, _ in mof_test])
    test_indices = np.array([i for i, _ in mof_test])

    # exclude zero-RMSD duplicate frameworks (a test frame identical to a training frame)
    ZERO_TOL = 1e-6
    is_dup = d_min < ZERO_TOL
    n_excluded = int(is_dup.sum())
    print(f"Excluding {n_excluded} zero-RMSD duplicate framework(s) out of {len(d_min)} test N02 frames")
    if n_excluded:
        print("  excluded test indices:", test_indices[is_dup].tolist())
    d_min_f, f_rmse_f, idx_f = d_min[~is_dup], f_rmse[~is_dup], test_indices[~is_dup]

    outlier_pos = np.argmin(np.abs(f_rmse_f - 23.0))
    outlier_idx = int(idx_f[outlier_pos])
    outlier_ct = test[outlier_idx].info.get("config_type")
    print(f"~23 meV/Å outlier frame: test index {outlier_idx}, config_type='{outlier_ct}', "
          f"force RMSE = {f_rmse_f[outlier_pos]:.2f} meV/Å, RMSD-to-train = {d_min_f[outlier_pos]:.3f} Å")

    # training-set internal spread for scale: RMSD of each train frame to its nearest *other* train frame
    sub = S_trn[:: max(1, len(S_trn) // 100)]
    nn_trn = []
    for k in range(len(sub)):
        ds = S_trn - sub[k][None]; ds -= np.round(ds)
        r = np.sqrt(np.mean(np.sum((ds @ cell) ** 2, axis=2), axis=1))
        nn_trn.append(np.sort(r)[1])
    nn_trn = np.array(nn_trn)

    fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.4))
    ax[0].hist(nn_trn, bins=25, alpha=0.6, label="train → nearest other train", density=True)
    ax[0].hist(d_min_f, bins=25, alpha=0.6, label="test → nearest train", density=True)
    ax[0].set_xlabel("framework RMSD (Å)"); ax[0].set_ylabel("density")
    ax[0].legend(frameon=False, fontsize=7.5)
    ax[0].text(-0.18, 1.05, "(a)", transform=ax[0].transAxes, fontsize=12, fontweight="bold", va="top")

    ax[1].scatter(d_min_f, f_rmse_f, s=18, c="tab:blue")
    ax[1].scatter([d_min_f[outlier_pos]], [f_rmse_f[outlier_pos]], s=50, facecolors="none",
                  edgecolors="red", linewidths=1.3, zorder=5)
    ax[1].set_xlabel("RMSD to nearest training frame (Å)")
    ax[1].set_ylabel("per-frame force RMSE (meV/Å)")
    if len(d_min_f) > 3:
        rho = np.corrcoef(d_min_f, f_rmse_f)[0, 1]
        ax[1].text(0.97, 0.03, f"Pearson r = {rho:.2f}", transform=ax[1].transAxes,
                   ha="right", va="bottom", fontsize=8)
    ax[1].text(-0.18, 1.05, "(b)", transform=ax[1].transAxes, fontsize=12, fontweight="bold", va="top")

    fig.tight_layout()
    savefig_dual(fig, "SI1_novelty")
    plt.show()
    print(f"median RMSD test→train (excl. dup) {np.median(d_min_f):.3f} Å ; "
          f"median train→train NN {np.median(nn_trn):.3f} Å")
    SUMMARY["T1_median_RMSD_test_to_train_A"] = float(np.median(d_min_f))
    SUMMARY["T1_median_RMSD_train_NN_A"] = float(np.median(nn_trn))
    SUMMARY["SI1_n_excluded_zero_rmsd_duplicates"] = n_excluded
    SUMMARY["SI1_outlier_test_index"] = outlier_idx
    SUMMARY["SI1_outlier_config_type"] = outlier_ct
else:
    print("Atom ordering or cell differs between frames; novelty analysis skipped.")


**Reading Test 1**
- Report the **test** RMSE/MAE by config type, not only the global number: `gas_N2` frames (2 atoms) have very different per-atom energies and should not dilute the framework statistics.
- A generalization gap (test/train RMSE) close to 1 means no over-fitting. If the novelty histogram shows test→train RMSD about equal to train→train nearest-neighbour RMSD, the test set is an interpolation test only; state this explicitly in the paper.
- A non-zero energy **bias** with small RMSE is an offset from `--E0s="average"`; it does not affect forces, MD or phonons, but matters for absolute adsorption energies.
- Typical force RMSEs reported for MOF-specific MLIPs are of order tens of meV/Å; compare with published models trained at the same DFT level rather than using a universal threshold.

---
## Test 2 — Phonon dispersion, phonon density of states and dynamical stability

Accurate forces (first derivatives) do not guarantee accurate curvature. Phonons use the second-derivative force constants and are the most sensitive standard test of the PES shape near the minimum.

**Protocol**
1. Take the lowest-DFT-energy `N02` frame, delete the two N₂ molecules → bare framework Zr₆O₃₂C₄₈H₂₈ (114 atoms), keeping the DFT cell (the only cell present in training).
2. Pre-relax the atomic positions with MACE, detect the space group with increasing tolerance, idealize the symmetry (`refine_symmetry`), then relax again under a `FixSymmetry` constraint to |F|max < 1×10⁻⁴ eV/Å. Hydroxylated UiO-66 is expected as F-43m (No. 216); the dehydroxylated/averaged form is Fm-3m (No. 225).
   Symmetry is not cosmetic here: without it Phonopy needs 3×114×2 = 684 displaced supercells instead of a few tens.

   **Precision:** this step runs on `calc_fp64` (CPU/float64, see the note in the header above). Relaxation converges to max|F| = 7.5×10⁻⁵ eV/Å (< 1×10⁻⁴ target, no oscillation) identically in float32/MPS and float64/CPU.

   **A reproducible ASE-vs-Phonopy symmetry discrepancy:** the ASE-side check above (`check_symmetry`/`refine_symmetry`, `symprec=1e-4`) reports **P1** (no symmetry) for this relaxed structure at every tolerance up to 0.1 Å, and the fallback constraint used for the final relax is therefore also effectively unconstrained. Yet Phonopy's own symmetry detection (`symprec=1e-3`, used when building the displaced supercells below) correctly finds **F-43m (216)** for both the 1×1×1 and 2×2×2 supercells, and generates only 34 displaced cells — the count expected for F-43m, not the 684 a true P1 cell would need. This is confirmed identical in float32/MPS and float64/CPU, so it is not a precision artifact: the relaxed structure is close to F-43m but outside ASE's tighter 1e-4/1e-3 tolerance, while Phonopy's 1e-3 tolerance and/or symmetrization algorithm recovers it. State this explicitly if reporting a space group in the manuscript — cite Phonopy's F-43m (216), not the ASE-side P1, and note the tolerance sensitivity.
3. Finite displacements with Phonopy (0.01 Å, ± displacements), residual forces of the undisplaced supercell subtracted, force constants symmetrized.
4. Band structure along the path chosen automatically by ASE for the face-centered cubic (FCC) Bravais lattice, density of states (DOS) on a Γ-centred mesh, and heat capacity (C_v) from the harmonic phonons — reported here as a single C_v(300 K) number rather than the full C_v(T) curve; only the 2×2×2 dispersion (Figure 2) and the 1×1×1-vs-2×2×2 overlay (SI-2) are plotted, both restricted to 0–100 cm⁻¹.
5. **Supercell convergence:** the site-energy receptive field is r_max × n_interactions = 10 Å, so interatomic force constants can extend up to ~20 Å. The primitive cell has a perpendicular width of only ≈12 Å, so 1×1×1 aliases long-range force constants. Both 1×1×1 and 2×2×2 are computed and overlaid; the difference is itself a reliability statement.

**Pass criteria used here:** no imaginary frequencies away from Γ; the three acoustic modes at Γ within ±0.05 THz (≈ ±1.7 cm⁻¹) of zero; bands converged between the two supercells.

**Limitations to state in any write-up:** (i) no non-analytical term correction, so LO–TO splitting at Γ is absent (this would require Born effective charges and the dielectric tensor from DFT); (ii) no DFT phonon reference was computed here — a direct comparison requires a DFT finite-displacement run on the same structure (the Phonopy yaml saved below is the starting point); (iii) harmonic phonons at 0 K do not include the anharmonicity present in 300 K MD.

In [ ]:
# ----------------------------- float64/CPU calculator for Test 2 and Test 4 -----------------------------
# Test 1's already-computed accuracy numbers stay on the fast MPS/float32 `calc` above (large N,
# statistical RMSE/MAE -- float32 noise is negligible there and Test 1 does not need rerunning).
# Symmetry-sensitive relaxation (Test 2) and small-strain energetics (Test 4) are a different regime:
# fmax convergence to 1e-4 eV/A and space-group detection can be corrupted by float32 rounding, so
# both use this full-precision CPU calculator instead, per the manuscript-precision requirement.
calc_fp64 = MACECalculator(model_paths=MODEL_PATH, device="cpu", default_dtype="float64")
print("Test 2 / Test 4 calculator: device=cpu, dtype=float64")


# NOTE ON PLACEMENT: constructing this MACECalculator sets torch's *global* default dtype to
# float64 (MACECalculator.__init__ calls torch_tools.set_default_dtype(default_dtype)), which breaks
# any later MPS/float32 tensor creation (the torch_geometric DataLoader batch build reads the global
# default dtype, and a float64 tensor cannot move to MPS). This cell must therefore run *after* Test 1
# (which needs the float32/MPS default) and *before* Test 2/4 (which need float64/CPU) -- not between
# cell 4 and Test 1 as originally placed, which crashed Test 1's evaluate() with exactly that error.

In [ ]:
# ----------------------------- build and relax the bare framework -----------------------------
mof_N02 = [a for a in train if a.info.get("config_type") == "N02"]
start = min(mof_N02, key=lambda a: get_reference(a)[0]).copy()
del start[[i for i, s in enumerate(start.get_chemical_symbols()) if s == "N"]]
print("framework formula:", start.get_chemical_formula(), "| natoms:", len(start))

STRUCTURE_FILE = None   # optional: path to an idealized CIF of the same primitive cell to use instead
if STRUCTURE_FILE is not None:
    start = read(STRUCTURE_FILE)

def spg(atoms, symprec):
    ds = check_symmetry(atoms, symprec=symprec)
    if ds is None:
        return None, None
    num = ds["number"] if isinstance(ds, dict) else ds.number
    sym = ds["international"] if isinstance(ds, dict) else ds.international
    return num, sym

fw = start.copy()
fw.calc = calc_fp64
BFGS(fw, logfile=os.path.join(OUT_DIR, "T2_prerelax.log")).run(fmax=0.02, steps=3000)

print("\nSpace group vs tolerance after pre-relaxation:")
found = None
for sp in [1e-4, 1e-3, 1e-2, 3e-2, 5e-2, 0.1, 0.2]:
    num, sym = spg(fw, sp)
    print(f"  symprec {sp:7.0e} Å -> {sym} ({num})")
    if found is None and num in (216, 225):
        found = sp
if found is None:
    warnings.warn("F-43m / Fm-3m not detected; continuing with the highest symmetry found. Check the structure.")
    found = 0.1

refine_symmetry(fw, symprec=found, verbose=True)
fw.set_constraint(FixSymmetry(fw, symprec=1e-4))
fw.calc = calc_fp64
BFGS(fw, logfile=os.path.join(OUT_DIR, "T2_symrelax.log")).run(fmax=1e-4, steps=10000)
fw.set_constraint()

F_res = np.abs(fw.get_forces()).max()
stress_GPa = fw.get_stress(voigt=False) / units.GPa
P_dft_cell = -np.trace(stress_GPa) / 3
num, sym = spg(fw, 1e-4)
print(f"\nfinal space group (symprec 1e-4): {sym} ({num}) | max |F| = {F_res:.2e} eV/Å")
print(f"MACE pressure at the DFT cell: {P_dft_cell:.3f} GPa  (diag stress GPa: {np.diag(stress_GPa).round(3)})")
fw_relaxed = fw.copy()
SUMMARY.update({"T2_space_group": f"{sym} ({num})", "T2_max_residual_force_eV_A": float(F_res),
                "MACE_pressure_at_DFT_cell_GPa": float(P_dft_cell)})

cellm = fw_relaxed.cell[:]
V = abs(np.linalg.det(cellm))
widths = [V / np.linalg.norm(np.cross(cellm[(i + 1) % 3], cellm[(i + 2) % 3])) for i in range(3)]
print(f"perpendicular widths of primitive cell: {np.round(widths, 2)} Å ; receptive field radius {RECEPTIVE_FIELD} Å")

In [ ]:
# ----------------------------- phonons -----------------------------
def to_phonopy(atoms):
    return PhonopyAtoms(symbols=atoms.get_chemical_symbols(), cell=atoms.cell[:],
                        scaled_positions=atoms.get_scaled_positions())

def to_ase(patoms):
    return Atoms(symbols=patoms.symbols, cell=patoms.cell, scaled_positions=patoms.scaled_positions, pbc=True)

def run_phonons(atoms, supercell, distance=0.01, symprec=1e-3):
    t0 = time.time()
    ph = Phonopy(to_phonopy(atoms), supercell_matrix=np.diag(supercell), primitive_matrix=None, symprec=symprec)
    ph.generate_displacements(distance=distance, is_plusminus=True)
    scs = ph.supercells_with_displacements
    print(f"supercell {supercell}: {len(ph.supercell)} atoms, {len(scs)} displaced cells, "
          f"space group {ph.symmetry.get_international_table()}")
    perfect = to_ase(ph.supercell); perfect.calc = calc_fp64
    F0 = perfect.get_forces()
    print(f"  max residual force in perfect supercell: {np.abs(F0).max():.2e} eV/Å (subtracted)")
    forces = []
    for k, sc in enumerate(scs):
        a = to_ase(sc); a.calc = calc_fp64
        forces.append(a.get_forces() - F0)
    ph.forces = np.array(forces)
    ph.produce_force_constants()
    ph.symmetrize_force_constants()
    tag = "x".join(map(str, supercell))
    ph.save(filename=os.path.join(OUT_DIR, f"T2_phonopy_params_{tag}.yaml"), settings={"force_constants": True})
    print(f"  done in {(time.time()-t0)/60:.1f} min")
    return ph

SUPERCELLS = [(1, 1, 1)] if QUICK else [(1, 1, 1), (2, 2, 2)]
phonons = {sc: run_phonons(fw_relaxed, sc) for sc in SUPERCELLS}

In [ ]:
# ----------------------------- band structure, DOS, stability analysis -----------------------------
path = fw_relaxed.cell.bandpath(npoints=300 if not QUICK else 120)
print("Bravais lattice detected by ASE:", path.cell.get_bravais_lattice().name, "| path:", path.path)
qpts = path.kpts                                     # fractional coordinates of this cell's reciprocal lattice
xq, xspecial, labels = path.get_linear_kpoint_axis()
labels = ["Γ" if l == "G" else l for l in labels]
q_cart = qpts @ fw_relaxed.cell.reciprocal()         # 1/Å (no 2π)
near_gamma = np.linalg.norm(q_cart, axis=1) < 0.01

MESH = [4, 4, 4] if QUICK else [8, 8, 8]
results = {}
for sc, ph in phonons.items():
    ph.run_band_structure([qpts], with_eigenvectors=False)
    fq = np.array(ph.get_band_structure_dict()["frequencies"][0])          # (nq, 3N) THz
    ph.run_mesh(MESH, with_eigenvectors=False, is_mesh_symmetry=True)
    mesh = ph.get_mesh_dict()
    fm, w = mesh["frequencies"], mesh["weights"]
    ph.run_total_dos(sigma=0.05, use_tetrahedron_method=False)   # sigma implies Gaussian smearing, not tetrahedron (phonopy>=3.5 needs this explicit)
    dos = ph.get_total_dos_dict()
    ph.run_thermal_properties(t_min=0, t_max=600, t_step=10)
    tp = ph.get_thermal_properties_dict()
    gamma = np.sort(ph.get_frequencies([0, 0, 0]))
    results[sc] = dict(fq=fq, dos=dos, tp=tp, gamma=gamma,
                       min_path_off_gamma=float(fq[~near_gamma].min()),
                       n_imag_mesh_frac=float(np.sum(w[:, None] * (fm < -0.05)) / (w.sum() * fm.shape[1])))
    print(f"\nsupercell {sc}:")
    print(f"  three lowest Γ frequencies: {gamma[:3].round(4)} THz  (acoustic, ideal = 0)")
    print(f"  4th–6th Γ frequencies     : {gamma[3:6].round(3)} THz = {(gamma[3:6]*THZ_TO_CM1).round(1)} cm^-1")
    print(f"  highest frequency         : {fq.max():.2f} THz = {fq.max()*THZ_TO_CM1:.0f} cm^-1")
    print(f"  most negative freq off Γ  : {results[sc]['min_path_off_gamma']:.4f} THz")
    print(f"  fraction of mesh modes < −0.05 THz: {results[sc]['n_imag_mesh_frac']:.2e}")

best = SUPERCELLS[-1]
R = results[best]
T = R["tp"]["temperatures"]
mass_g_mol = fw_relaxed.get_masses().sum()
Cv_300 = np.interp(300, T, R["tp"]["heat_capacity"]) / mass_g_mol     # J/(g K)
print(f"\nHarmonic C_v(300 K) = {Cv_300:.4f} J g^-1 K^-1  (supercell {best})")
acoustic_ok = np.all(np.abs(R["gamma"][:3]) < 0.05)
stable = R["min_path_off_gamma"] > -0.05
print("acoustic sum rule satisfied (|ω|<0.05 THz):", acoustic_ok, "| no imaginary modes off Γ:", stable)

if len(SUPERCELLS) > 1:
    f1, f2 = results[SUPERCELLS[0]]["fq"], results[SUPERCELLS[1]]["fq"]
    conv = np.abs(np.sort(f1, axis=1) - np.sort(f2, axis=1))
    print(f"supercell convergence 1x1x1 vs 2x2x2: mean |Δω| = {conv.mean():.3f} THz, max = {conv.max():.3f} THz")
    SUMMARY["T2_band_convergence_mean_dTHz"] = float(conv.mean())
SUMMARY.update({"T2_supercell": "x".join(map(str, best)), "T2_gamma_acoustic_THz": R["gamma"][:3].tolist(),
                "T2_min_freq_off_gamma_THz": R["min_path_off_gamma"], "T2_dynamically_stable": bool(stable),
                "T2_Cv_300K_J_gK": float(Cv_300)})

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Figure 2 — Phonon dispersion (2×2×2 supercell, full precision, float64/CPU)
# ══════════════════════════════════════════════════════════════════════════
best = SUPERCELLS[-1]   # (2, 2, 2)
f_best = results[best]["fq"] * THZ_TO_CM1

fig, ax = plt.subplots(figsize=(5, 5))
for j in range(f_best.shape[1]):
    ax.plot(xq, f_best[:, j], color="0.15", lw=0.8)
for x in xspecial:
    ax.axvline(x, color="k", lw=0.5)
ax.axhline(0, color="0.5", lw=0.5)
ax.set_xticks(xspecial); ax.set_xticklabels(labels)
ax.set_xlim(xq[0], xq[-1])
ax.set_ylim(0, 100)
ax.set_ylabel("Frequency (cm$^{-1}$)", fontweight="bold")
fig.tight_layout()
savefig_dual(fig, "Fig_MACE_phonon_dispersion")
plt.show()
print("Saved Fig_MACE_phonon_dispersion.png/.pdf ->", FIG_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SI-2 — Phonon dispersion, 1×1×1 vs 2×2×2 supercell (finite-size convergence)
# ══════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(5, 5))
styles = {SUPERCELLS[0]: dict(color="0.6", lw=0.8, ls="--"), best: dict(color="tab:blue", lw=0.9, ls="-")}
for sc in SUPERCELLS:
    f = results[sc]["fq"] * THZ_TO_CM1
    for j in range(f.shape[1]):
        ax.plot(xq, f[:, j], label="×".join(map(str, sc)) if j == 0 else None, **styles[sc])
for x in xspecial:
    ax.axvline(x, color="k", lw=0.5)
ax.axhline(0, color="0.5", lw=0.5)
ax.set_xticks(xspecial); ax.set_xticklabels(labels)
ax.set_xlim(xq[0], xq[-1])
ax.set_ylim(0, 100)
ax.set_ylabel("Frequency (cm$^{-1}$)", fontweight="bold")
ax.legend(title="supercell", fontsize=8, frameon=False)
fig.tight_layout()
savefig_dual(fig, "SI2_dispersion_1x1x1_vs_2x2x2")
plt.show()

conv_cm1 = np.abs(np.sort(results[SUPERCELLS[0]]["fq"], axis=1) - np.sort(results[best]["fq"], axis=1)) * THZ_TO_CM1
print(f"convergence 1x1x1 vs 2x2x2 (cm^-1): mean |Δω| = {conv_cm1.mean():.3f}, max = {conv_cm1.max():.3f}")
print("Saved SI2_dispersion_1x1x1_vs_2x2x2.png/.pdf ->", FIG_DIR)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FTIR proxy from the 2×2×2 phonon DOS (curve only, no bands/legend)
# ══════════════════════════════════════════════════════════════════════════
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

def phonon_dos_to_ftir(freq_THz, dos, wn_range=(400, 4000), n_pts=3600, smooth_sigma=8):
    """Same method as mace_uio66_ftir.py / MOF_Characterization.ipynb: interpolate the
    DOS onto a uniform cm^-1 grid, Gaussian-smooth, normalize to 0-100 as an absorbance
    proxy. Peak *positions* are the meaningful output (no dipole-derivative weighting)."""
    THz_to_cm = 33.356
    wn_data = np.clip(freq_THz, 0, None) * THz_to_cm
    order = np.argsort(wn_data)
    wn_data, dos_data = wn_data[order], dos[order]
    wn = np.linspace(*wn_range, n_pts)
    spec = np.interp(wn, wn_data, dos_data, left=0, right=0)
    spec = gaussian_filter1d(spec, sigma=smooth_sigma)
    if spec.max() > 0:
        spec = spec / spec.max() * 100.0
    pidx, _ = find_peaks(spec, height=12, distance=25)
    peaks = sorted([{"wn": wn[i], "intensity": spec[i] / 100.0} for i in pidx], key=lambda x: x["wn"])
    return wn, spec, peaks

dos_2x2x2 = results[SUPERCELLS[-1]]["dos"]
wn, spec, peaks = phonon_dos_to_ftir(dos_2x2x2["frequency_points"], dos_2x2x2["total_dos"])
trans = 100.0 - gaussian_filter1d(spec, sigma=3)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(wn, trans, color="#8E24AA", lw=1.8)
ax.set_xlabel("Wavenumber (cm$^{-1}$)", fontsize=12, fontweight="bold")
ax.set_ylabel("Transmittance (a.u.)", fontsize=12, fontweight="bold")
ax.set_xlim(4000, 400)
ax.set_ylim(0, 105)
ax.set_yticks([])
ax.grid(False)
for sp in ax.spines.values():
    sp.set_linewidth(1.6)
ax.tick_params(width=1.3, length=5)
fig.tight_layout()
savefig_dual(fig, "FTIR_MACE_2x2x2")
plt.show()
print(f"{len(peaks)} FTIR-window peaks: " + ", ".join(f"{p['wn']:.0f}" for p in peaks) + " cm^-1")
print("Saved FTIR_MACE_2x2x2.png/.pdf ->", FIG_DIR)


**Reading Test 2**
- *Imaginary modes away from Γ* indicate that the relaxed structure is a saddle point of the MACE PES → unreliable curvature (or an insufficiently relaxed/symmetrized structure; check `T2_symrelax.log` first).
- *Small negative acoustic frequencies at Γ* (|ω| < 0.05 THz) are a numerical artefact, not an instability.
- The 1×1×1 vs 2×2×2 difference quantifies the error of the finite-supercell approximation. If it is large at low frequency, go to 2×2×2 for publication (and consider 3×3×3 only on GPU with enough memory).
- Figure 2 / SI-2 show the dispersion only (no DOS or Γ-point stick panel in the manuscript figures). For an FTIR comparison, read Γ-point mode positions directly from the dispersion at q=Γ (x=0 in Figure 2) or from `mace_uio66_ftir.py`'s output (`UiO66_MACE_FTIR.png`, used in `MOF_Characterization.ipynb`): carboxylate stretching ~1400–1600 cm⁻¹, μ₃-O/Zr–O region below ~800 cm⁻¹. Neither is a real IR spectrum (no dipole derivatives), so compare band **positions** only, not intensities.
- MOFSimBench reports that many universal MLIPs over-predict C_v because they under-predict PES curvature; a C_v(300 K) far above the DFT value for UiO-66 from that benchmark would point to the same issue. Take the DFT reference value from the paper's SI rather than a figure reading.
- To compare directly with DFT: run VASP on the displaced supercells written by `phonopy --load T2_phonopy_params_2x2x2.yaml` (same displacements), then overlay both band structures.

---
## Test 3 — Energy conservation in microcanonical (NVE) molecular dynamics

MACE forces are exact negative gradients of the MACE energy (automatic differentiation), so in NVE the total energy should be conserved up to integrator error. Violations reveal a non-smooth PES (e.g. discontinuities at the cutoff, holes in regions not covered by training) or instability — exactly the failure modes that would silently corrupt the MSD/diffusion production runs.

**Protocol** (on the system the model was trained for: UiO-66 primitive cell + 2 N₂, 118 atoms, same cell as training):
1. Maxwell–Boltzmann velocities at 300 K; Langevin (NVT) equilibration, 2 ps, Δt = 0.5 fs.
2. From the *same* equilibrated state, two NVE runs of equal physical length (10 ps) with Δt = 0.5 fs and Δt = 0.25 fs (velocity Verlet).
3. Metrics: linear drift of total energy (meV/atom/ps); ratio of total-energy to potential-energy fluctuations σ(E_tot)/σ(E_pot); maximum force and N–N bond length (guest integrity) along the trajectory.
**This section is off by default (`RUN_TEST3 = False`)** — Test 3B below reruns the identical check on the real LAMMPS/`-lammps.pt` production path, at both 118 and 944 atoms, and is already populated with real results; set `RUN_TEST3 = True` only if you specifically want the ASE/`.model` cross-check described here.

4. **Time-step scaling check:** for a smooth, conservative PES the shadow-Hamiltonian fluctuation of velocity Verlet scales as Δt². Halving Δt should reduce σ(E_tot) by ≈4×. Much weaker scaling points to PES roughness rather than integration error.

In [ ]:
# Test 3 duplicates Test 3B (real LAMMPS NVE, both system sizes, already run) with a slower
# ASE/CPU implementation on a single 118-atom cell. Left in for anyone who wants the direct
# ASE/.model vs LAMMPS/-lammps.pt cross-check (see Reading Test 3B); off by default so
# "Run All" does not spend ~minutes-to-hours here redundantly.
RUN_TEST3 = False

if RUN_TEST3:
    torch.set_default_dtype(torch.float32)   # undo calc_fp64's global float64 default (Test 2/4) so
                                              # this section's mps/float32 `calc` works again
    T_MD = 300.0
    N_EQ_STEPS = 400 if QUICK else 4000              # 0.2 / 2 ps at 0.5 fs
    NVE_TIME_PS = 1.0 if QUICK else 10.0
    SAMPLE_FS = 5.0

    md_start = min([a for a in test if a.info.get("config_type") == "N02"],
                   key=lambda a: get_reference(a)[0]).copy()
    md_start.calc = calc
    N_idx = [i for i, s in enumerate(md_start.get_chemical_symbols()) if s == "N"]
    # pair N atoms into molecules by nearest partner (minimum image)
    D = md_start.get_all_distances(mic=True)
    pairs = []
    left = list(N_idx)
    while left:
        i = left.pop(0)
        j = min(left, key=lambda k: D[i, k]); left.remove(j); pairs.append((i, j))
    print("N2 molecules (atom indices):", pairs, "| initial N–N:", [round(D[i, j], 4) for i, j in pairs])

    MaxwellBoltzmannDistribution(md_start, temperature_K=T_MD, rng=rng)
    Stationary(md_start)
    t0 = time.time()
    Langevin(md_start, timestep=0.5 * units.fs, temperature_K=T_MD, friction=0.01 / units.fs, rng=rng).run(N_EQ_STEPS)
    print(f"equilibration done in {time.time()-t0:.0f} s, T = {md_start.get_temperature():.1f} K")
    eq_state = md_start.copy()          # copies positions and momenta
else:
    print("RUN_TEST3 = False -> skipped (redundant with Test 3B's real LAMMPS NVE runs)")


In [ ]:
if RUN_TEST3:
    def run_nve(state, dt_fs, total_ps):
        a = state.copy(); a.calc = calc
        nsteps = int(round(total_ps * 1000 / dt_fs))
        every = max(1, int(round(SAMPLE_FS / dt_fs)))
        log = defaultdict(list)
        dyn = VelocityVerlet(a, timestep=dt_fs * units.fs)
        def record():
            epot = a.get_potential_energy(); ekin = a.get_kinetic_energy()
            log["t_ps"].append(dyn.nsteps * dt_fs / 1000); log["epot"].append(epot); log["ekin"].append(ekin)
            log["etot"].append(epot + ekin); log["T"].append(a.get_temperature())
            log["fmax"].append(np.abs(a.get_forces()).max())
            log["dNN"].append([a.get_distance(i, j, mic=True) for i, j in pairs])
        dyn.attach(record, interval=every)
        record_t0 = time.time()
        dyn.run(nsteps)
        log = {k: np.array(v) for k, v in log.items()}
        print(f"NVE dt={dt_fs} fs: {nsteps} steps in {(time.time()-record_t0)/60:.1f} min")
        return log

    def nve_metrics(log, natoms):
        t, et, ep = log["t_ps"], log["etot"], log["epot"]
        slope = np.polyfit(t, et, 1)[0]
        resid = et - np.polyval(np.polyfit(t, et, 1), t)
        return dict(drift_meV_atom_ps=1000 * slope / natoms,
                    sigma_etot_meV_atom=1000 * resid.std() / natoms,
                    ratio_sigma_tot_pot=resid.std() / ep.std(),
                    max_dev_meV_atom=1000 * np.abs(et - et[0]).max() / natoms,
                    T_mean=log["T"].mean(), fmax_max=log["fmax"].max(),
                    dNN_range=(log["dNN"].min(), log["dNN"].max()))

    nat = len(eq_state)
    logs = {dt: run_nve(eq_state, dt, NVE_TIME_PS) for dt in (0.5, 0.25)}
    mets = {dt: nve_metrics(l, nat) for dt, l in logs.items()}
    for dt, m in mets.items():
        print(f"\nΔt = {dt} fs")
        for k, v in m.items():
            print(f"  {k:22s}: {v}")
    scaling = mets[0.5]["sigma_etot_meV_atom"] / mets[0.25]["sigma_etot_meV_atom"]
    print(f"\nσ(E_tot) ratio (0.5 fs / 0.25 fs) = {scaling:.2f}   (ideal Verlet ≈ 4)")
    SUMMARY.update({"T3_drift_meV_atom_ps_dt0.5": mets[0.5]["drift_meV_atom_ps"],
                    "T3_drift_meV_atom_ps_dt0.25": mets[0.25]["drift_meV_atom_ps"],
                    "T3_sigma_ratio_tot_pot_dt0.5": mets[0.5]["ratio_sigma_tot_pot"],
                    "T3_dt_scaling_ratio": scaling, "T3_T_mean_K": mets[0.5]["T_mean"],
                    "T3_NN_bond_range_A": [float(x) for x in mets[0.5]["dNN_range"]]})
else:
    print("RUN_TEST3 = False -> skipped")


In [ ]:
if RUN_TEST3:
    fig, axs = plt.subplots(2, 2, figsize=(11, 7))
    c = {0.5: "tab:blue", 0.25: "tab:orange"}
    for dt, l in logs.items():
        e0 = l["etot"][0]
        axs[0, 0].plot(l["t_ps"], 1000 * (l["etot"] - e0) / nat, color=c[dt], lw=0.8,
                       label=f"Δt={dt} fs, drift {mets[dt]['drift_meV_atom_ps']:.3f} meV/atom/ps")
    axs[0, 0].set_xlabel("t (ps)"); axs[0, 0].set_ylabel("E$_{tot}$ − E$_{tot}$(0) (meV/atom)")
    axs[0, 0].set_title("(a) Total-energy conservation"); axs[0, 0].legend(fontsize=8, frameon=False)

    l = logs[0.5]
    axs[0, 1].plot(l["t_ps"], 1000 * (l["epot"] - l["epot"].mean()) / nat, lw=0.6, color="0.5", label="E$_{pot}$")
    axs[0, 1].plot(l["t_ps"], 1000 * (l["etot"] - l["etot"].mean()) / nat, lw=0.8, color="tab:blue", label="E$_{tot}$")
    axs[0, 1].set_xlabel("t (ps)"); axs[0, 1].set_ylabel("meV/atom (mean removed)")
    axs[0, 1].set_title(f"(b) Fluctuations, σ$_{{tot}}$/σ$_{{pot}}$ = {mets[0.5]['ratio_sigma_tot_pot']:.3f}")
    axs[0, 1].legend(fontsize=8, frameon=False)

    axs[1, 0].plot(l["t_ps"], l["T"], lw=0.6)
    axs[1, 0].axhline(T_MD, color="k", ls=":", lw=0.8)
    axs[1, 0].set_xlabel("t (ps)"); axs[1, 0].set_ylabel("T (K)"); axs[1, 0].set_title("(c) Instantaneous temperature (NVE)")

    for k in range(l["dNN"].shape[1]):
        axs[1, 1].plot(l["t_ps"], l["dNN"][:, k], lw=0.6, label=f"N₂ #{k+1}")
    axs[1, 1].set_xlabel("t (ps)"); axs[1, 1].set_ylabel("N–N distance (Å)")
    axs[1, 1].set_title("(d) Guest integrity"); axs[1, 1].legend(fontsize=8, frameon=False)
    fig.tight_layout(); savefig(fig, "T3_nve.png"); plt.show()
else:
    print("RUN_TEST3 = False -> skipped")


**Reading Test 3**
- No universal threshold exists; report the numbers. For a conservative MLIP with a sub-femtosecond time step, the drift is usually well below 1 meV/atom/ps and σ(E_tot)/σ(E_pot) is of order 10⁻²; a steady trend in (a) or a σ ratio (0.5/0.25 fs) far below 4 indicates PES roughness.
- (d) should oscillate around ~1.10 Å (the N–N distance in the DFT data spans 1.074–1.158 Å). Dissociation or a steadily growing force maximum means the trajectory has entered a region without training coverage.
- The 118-atom cell is smaller than twice the receptive field; this is consistent with how the model was trained, but production runs in larger supercells change the environment statistics only beyond ~10 Å and are expected to be at least as stable. Repeating this test in the LAMMPS production supercell with `-lammps.pt` (`fix nve`, `thermo_style custom step pe ke etotal`) is a cheap extra check.

---
## Test 3B — Production-scale LAMMPS NVE: 1×1×1 vs 2×2×2 finite-size check

Test 3 runs NVE inside this notebook with ASE on the CPU, on the exact 118-atom cell the
model was trained for. Production MD (MSD/diffusion) will run in **LAMMPS** on a larger
supercell with the compiled `-lammps.pt` model, so the same conservation check needs to be
repeated there — and repeated at at least two system sizes to see whether it is
size-converged before trusting the larger production runs.

**Protocol** (`in.equil` / `in.nve`, `pair_style mace no_domain_decomposition`, UiO-66
primitive cell + 2 N₂, replicated by the factor below):
1. NVT (Langevin) equilibration at 300 K, 2 ps, Δt = 0.5 fs → `equil_300K.dat`, `eq_300K.data`.
2. NVE production from the equilibrated state, 80 ps, Δt = 0.5 fs → `nve_dt0.5.dat`,
   `N_dt0.5.lammpstrj` (guest atoms only), `traj_dt0.5.lammpstrj` (all atoms).
3. The same metrics as Test 3 (drift, σ(E_tot)/σ(E_pot), temperature stability, max force,
   N–N bond range) computed independently from the LAMMPS output, plus a structural-eligibility
   check on the equilibrated cell (atom count, no NaN/overlaps, composition, density).
4. **Finite-size check:** the 1×1×1 cell (118 atoms, same cell as training) is repeated with
   `replicate 2 2 2` (944 atoms) so guest molecules and framework atoms sit ≥ one extra
   receptive-field radius (`RECEPTIVE_FIELD` = R_MAX × N_INTERACTIONS Å, see Test 1 setup)
   from their periodic images in every direction. Per-atom drift/fluctuation metrics should be
   essentially size-independent if the potential is well-behaved; a mismatch that *grows* with
   system size (rather than shrinking or staying flat) points at an artifact specific to the
   small training cell rather than at ordinary statistical noise.

The cells below search the `1*1*1/` and `2*2*2/` folders for these files and analyze whichever
is present — rerun this section once the `2*2*2/` run finishes; nothing above or below this
section depends on it.


In [ ]:
import glob

LAMMPS_TAGS = {"1x1x1": "1*1*1", "2x2x2": "2*2*2"}
REQUIRED_FILES = ["eq_300K.data", "equil_300K.dat", "nve_dt0.5.dat", "N_dt0.5.lammpstrj",
                  "log_equil.lammps", "log_nve_dt0.5.lammps"]

def find_lammps_run(folder_name):
    """Folder names contain literal '*' characters -> glob.escape before globbing.
    Looks next to the notebook first, then anywhere below it (in case the folders get
    moved into a subdirectory later)."""
    here = glob.glob(os.path.join(glob.escape(folder_name), "*"))
    nested = glob.glob(os.path.join("**", glob.escape(folder_name), "*"), recursive=True)
    files = {}
    for f in here + nested:
        files.setdefault(os.path.basename(f), f)
    return files

lammps_runs = {}
print("Searching for LAMMPS equilibration/NVE output ...")
for tag, folder in LAMMPS_TAGS.items():
    files = find_lammps_run(folder)
    have = [f for f in REQUIRED_FILES if f in files]
    missing = [f for f in REQUIRED_FILES if f not in files]
    status = "COMPLETE" if not missing else f"INCOMPLETE ({len(missing)} file(s) missing)"
    print(f"  {tag:6s} ('{folder}/'): {status}")
    if missing:
        print(f"           missing: {missing}")
    if have:
        lammps_runs[tag] = files

print(f"\nSystems with at least some output: {list(lammps_runs)}")
if "2x2x2" not in lammps_runs:
    print("2x2x2 not found yet (or still running) -> only 1x1x1 will be analyzed below; "
          "rerun this section once it lands in '2*2*2/'.")


In [ ]:
def read_thermo_dat(path):
    """Parses the '# col1 col2 ...' whitespace tables written by in.equil / in.nve."""
    with open(path) as fh:
        header = fh.readline().lstrip("#").split()
    data = np.loadtxt(path, comments="#")
    if data.ndim == 1:
        data = data[None, :]
    return {name: data[:, i] for i, name in enumerate(header)}

def read_natoms_from_lammps_data(path):
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line.endswith(" atoms"):
                return int(line.split()[0])
    raise ValueError(f"atom count not found in {path}")

def lammps_run_completed(log_path):
    """'Total wall time' is only written once LAMMPS finishes; its absence means the
    job is still running (or crashed/was killed) -- do not trust partial output as if final."""
    with open(log_path) as fh:
        text = fh.read()
    return "Total wall time" in text and "ERROR" not in text

def nve_metrics_from_dat(cols, natoms):
    t, et, ep = cols["time_ps"], cols["etot_eV"], cols["pe_eV"]
    fit = np.polyfit(t, et, 1)
    resid = et - np.polyval(fit, t)
    return dict(
        drift_meV_atom_ps=1000 * fit[0] / natoms,
        sigma_etot_meV_atom=1000 * resid.std() / natoms,
        ratio_sigma_tot_pot=float(resid.std() / ep.std()),
        max_dev_meV_atom=1000 * np.abs(et - et[0]).max() / natoms,
        T_mean_K=float(cols["T_K"].mean()), T_std_K=float(cols["T_K"].std()),
        fmax_max_eV_A=float(cols["fmax_eV_A"].max()) if "fmax_eV_A" in cols else None,
        n_samples=int(len(t)), total_ps=float(t[-1] - t[0]),
    )

def dNN_from_trajectory(path):
    """Pairs N atoms into molecules once (nearest partner, frame 0), then follows the
    same index pair through the whole trajectory -- LAMMPS `dump_modify sort id` keeps
    atom order fixed across frames, so this is safe without re-pairing every frame."""
    frames = read(path, format="lammps-dump-text", index=":")
    a0 = frames[0]
    D0 = a0.get_all_distances(mic=True)
    left = list(range(len(a0)))
    pairs = []
    while left:
        i = left.pop(0)
        j = min(left, key=lambda k: D0[i, k]); left.remove(j); pairs.append((i, j))
    dNN = np.array([[a.get_distance(i, j, mic=True) for i, j in pairs] for a in frames])
    return dict(pairs=pairs, dNN=dNN, n_molecules=len(pairs), n_frames=len(frames))

LAMMPS_Z_OF_TYPE = {1: 6, 2: 1, 3: 7, 4: 8, 5: 40}   # 1 C, 2 H, 3 N, 4 O, 5 Zr (see in.equil)

def structural_eligibility(eq_data_path, expected_natoms=None):
    """Sanity checks on the equilibrated cell before trusting anything computed from it."""
    natoms = read_natoms_from_lammps_data(eq_data_path)
    atoms = read(eq_data_path, format="lammps-data", Z_of_type=LAMMPS_Z_OF_TYPE, atom_style="atomic")
    pos = atoms.get_positions()
    dmat = atoms.get_all_distances(mic=True)
    np.fill_diagonal(dmat, np.inf)
    min_d = float(dmat.min())
    checks = dict(
        natoms=natoms,
        natoms_as_expected=bool(expected_natoms is None or natoms == expected_natoms),
        has_nan_or_inf=bool(not np.all(np.isfinite(pos))),
        cell_volume_A3=float(atoms.get_volume()),
        density_atoms_per_A3=natoms / atoms.get_volume(),
        min_interatomic_distance_A=min_d,
        no_overlapping_atoms=bool(min_d > 0.6),           # well below any real covalent bond
        formula=atoms.get_chemical_formula(),
    )
    checks["eligible"] = bool(checks["natoms_as_expected"] and not checks["has_nan_or_inf"]
                              and checks["no_overlapping_atoms"])
    return checks


In [ ]:
system_results = {}
for tag, files in lammps_runs.items():
    print(f"\n=== {tag} ===")
    eq_done = ("log_equil.lammps" in files) and lammps_run_completed(files["log_equil.lammps"])
    nve_done = ("log_nve_dt0.5.lammps" in files) and lammps_run_completed(files["log_nve_dt0.5.lammps"])
    print(f"  equilibration finished: {eq_done} | NVE production finished: {nve_done}")
    if not (eq_done and nve_done and "eq_300K.data" in files and "nve_dt0.5.dat" in files
            and "equil_300K.dat" in files and "N_dt0.5.lammpstrj" in files):
        print("  -> run not complete yet, skipping metric computation for this system")
        continue

    natoms = read_natoms_from_lammps_data(files["eq_300K.data"])
    eq_cols = read_thermo_dat(files["equil_300K.dat"])
    nve_cols = read_thermo_dat(files["nve_dt0.5.dat"])
    metrics = nve_metrics_from_dat(nve_cols, natoms)
    metrics["equil_T_final_K"] = float(eq_cols["T_K"][-1])
    metrics["equil_T_mean_last_half_K"] = float(eq_cols["T_K"][len(eq_cols["T_K"]) // 2:].mean())

    nn = dNN_from_trajectory(files["N_dt0.5.lammpstrj"])
    metrics["N2_pairs"] = nn["pairs"]
    metrics["n_N2_molecules"] = nn["n_molecules"]
    metrics["dNN_range_A"] = [float(nn["dNN"].min()), float(nn["dNN"].max())]
    metrics["N2_dissociated"] = bool(nn["dNN"].max() > 2.0)   # covalent N-N ~1.1 A

    struct = structural_eligibility(files["eq_300K.data"], expected_natoms=natoms)
    metrics["structural_eligibility"] = struct

    system_results[tag] = dict(natoms=natoms, metrics=metrics)
    print(f"  natoms={natoms} | drift={metrics['drift_meV_atom_ps']:.4g} meV/atom/ps | "
          f"sigma(Etot)={metrics['sigma_etot_meV_atom']:.4g} meV/atom | "
          f"sigma_tot/sigma_pot={metrics['ratio_sigma_tot_pot']:.4g} | "
          f"T={metrics['T_mean_K']:.1f}+/-{metrics['T_std_K']:.1f} K | "
          f"fmax={metrics['fmax_max_eV_A']:.2f} eV/A | "
          f"dNN=[{metrics['dNN_range_A'][0]:.3f}, {metrics['dNN_range_A'][1]:.3f}] A | "
          f"eligible={struct['eligible']}")

json_path = os.path.join(OUT_DIR, "T3B_lammps_1x1x1_vs_2x2x2_metrics.json")
with open(json_path, "w") as f:
    json.dump(system_results, f, indent=2, default=float)
print(f"\nsaved per-system metrics -> {json_path}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SI-4 — LAMMPS NVE energy conservation: 1-ps block-averaged drift
# ══════════════════════════════════════════════════════════════════════════
if "1x1x1" in system_results and "2x2x2" in system_results:
    fig, ax = plt.subplots(figsize=(4.2, 3.4))
    colors_si4 = {"1x1x1": "tab:blue", "2x2x2": "tab:orange"}
    block_ps = 1.0
    si4_drift = {}
    for tag in ("1x1x1", "2x2x2"):
        nat = system_results[tag]["natoms"]
        cols = read_thermo_dat(lammps_runs[tag]["nve_dt0.5.dat"])
        t, etot = cols["time_ps"], cols["etot_eV"]
        nblocks = int(np.floor(t[-1] / block_ps))
        t_block, e_block = [], []
        for b in range(nblocks):
            m = (t >= b * block_ps) & (t < (b + 1) * block_ps)
            if not np.any(m):
                continue
            t_block.append((b + 0.5) * block_ps)
            e_block.append(etot[m].mean())
        t_block, e_block = np.array(t_block), np.array(e_block)
        de_ueV_atom = (e_block - e_block[0]) / nat * 1e6   # eV -> ueV, per atom
        slope, _ = np.polyfit(t_block, de_ueV_atom, 1)
        si4_drift[tag] = float(slope)
        ax.plot(t_block, de_ueV_atom, "o-", ms=3, lw=1.0, color=colors_si4[tag],
                label=f"{tag} ({nat} atoms)\ndrift = {slope:.2e} μeV/atom/ps")
    ax.set_xlabel("t (ps)")
    ax.set_ylabel(r"$\langle E_{tot}\rangle_{1\,\mathrm{ps}} - E_{tot}(0)$ ($\mu$eV/atom)")
    ax.legend(fontsize=6.5, frameon=False, loc="best")
    fig.tight_layout()
    savefig_dual(fig, "SI4_nve_drift")
    plt.show()
    print("Saved SI4_nve_drift.png/.pdf ->", FIG_DIR)
    SUMMARY.update({"SI4_drift_ueV_atom_ps_1x1x1": si4_drift["1x1x1"],
                    "SI4_drift_ueV_atom_ps_2x2x2": si4_drift["2x2x2"]})
else:
    print(f"Need both 1x1x1 and 2x2x2 to plot SI-4; currently analyzed: {list(system_results)}")


**Reading Test 3B**
- These numbers are computed independently of Test 3 (different code path: real LAMMPS +
  the compiled `-lammps.pt` model vs. ASE + the `.model` file in this notebook). If Test 3's
  118-atom ASE run and Test 3B's 118-atom LAMMPS run (`1x1x1`) disagree by orders of magnitude
  in drift or σ(E_tot)/σ(E_pot), suspect an ASE/LAMMPS precision or unit mismatch (e.g. `float32`
  vs `float64`, or a different `r_max`/neighbor list) rather than the model itself.
- The **replication check** must give exactly 8.0 (2×2×2 = 8 primitive cells); anything else
  means `in.equil`'s `replicate` line, or the folder contents, do not match what the folder name
  claims, and everything below it should not be trusted until that is fixed.
- If `T3B_size_convergence_ok` is `False`, look at *which* per-atom quantity actually diverges
  between 1×1×1 and 2×2×2 (density should match to a small fraction of a percent by
  construction — a bigger mismatch there means the two `.data` files were not built from the
  same relaxed structure, not a physics problem). A genuine drift/σ(E_tot) blow-up at 2×2×2
  that is *larger* than at 1×1×1 is a stronger flag: it says the small training/eval cell was
  hiding an instability that only appears once periodic images are far enough apart.
- `structural_eligibility.eligible = False` on either system should block interpreting anything
  else about that run (drift, phonons downstream, etc.) until the underlying `.data` file is
  regenerated — it means the equilibrated structure itself failed a basic sanity check (wrong
  atom count, NaN coordinates, or atoms overlapping).


---
## Test 4 — Equation of state and bulk modulus (extrapolation test)

All MOF frames in the training data share **one** cell volume (2278.6 Å³) and no stress was fitted, so the energy–volume curve is pure extrapolation. This is the most demanding of the four tests.

**Protocol**
1. Start from the symmetrized, relaxed framework of Test 2.
2. Scale the cell isotropically to V/V_DFT = 0.94 … 1.06 (13 points; ±2 % linear strain). At each volume relax atomic positions under `FixSymmetry` to |F|max < 10⁻³ eV/Å.
3. Fit the third-order Birch–Murnaghan (BM3) EOS to E(V) → V₀, B₀, B₀′.
4. **Internal consistency:** independently fit BM3 P(V) to the virial pressure P = −tr(σ)/3 from MACE stresses. Energy- and stress-derived B₀ must agree; disagreement indicates a stress-implementation or numerical problem.
5. Convexity: the discrete second derivative d²E/dV² must be positive over the whole range.

**Literature context.** The DFT reference used throughout this notebook (Table 1) is Wu, Yildirim & Zhou, *J. Phys. Chem. Lett.* **2013**, 4, 925: B₀ = 41.0 GPa and single-crystal elastic constants C₁₁/C₁₂/C₄₄ = 59.35/31.85/17.63 GPa for UiO-66, from DFT. MOFSimBench (npj Comput. Mater. 2025) separately finds that most universal MLIPs underestimate MOF bulk moduli; that trend, not a specific number from it, is the relevant comparison here. Because our DFT level (PBE+D3(BJ)) and cell may differ from Wu et al.'s, agreement within a few GPa is the meaningful criterion, not an exact match — see Table 1 for the actual MACE-vs-DFT numbers.

In [ ]:
SCALES = np.linspace(0.94, 1.06, 7 if QUICK else 13)
V_dft = fw_relaxed.get_volume()
Vs, Es, Ps = [], [], []
for s in SCALES:
    a = fw_relaxed.copy()
    a.set_constraint()
    a.set_cell(fw_relaxed.cell[:] * s ** (1 / 3), scale_atoms=True)
    a.calc = calc_fp64
    a.set_constraint(FixSymmetry(a, symprec=1e-4))
    BFGS(a, logfile=os.path.join(OUT_DIR, "T4_eos_relax.log")).run(fmax=1e-3, steps=3000)
    a.set_constraint()
    Vs.append(a.get_volume()); Es.append(a.get_potential_energy())
    Ps.append(-np.trace(a.get_stress(voigt=False)) / 3 / units.GPa)
    print(f"V/V_DFT = {s:.3f}  V = {Vs[-1]:9.3f} Å^3  E = {Es[-1]:.5f} eV  P = {Ps[-1]:7.3f} GPa")
Vs, Es, Ps = map(np.array, (Vs, Es, Ps))

In [ ]:
EV_A3_TO_GPA = 1.0 / units.GPa      # eV/Å^3 -> GPa

def bm3_E(V, E0, V0, B0, Bp):
    eta = (V0 / V) ** (2 / 3)
    return E0 + 9 * V0 * B0 / 16 * ((eta - 1) ** 3 * Bp + (eta - 1) ** 2 * (6 - 4 * eta))

def bm3_P(V, V0, B0, Bp):
    x = (V0 / V)
    return 1.5 * B0 * (x ** (7 / 3) - x ** (5 / 3)) * (1 + 0.75 * (Bp - 4) * (x ** (2 / 3) - 1))

# energy fit (B0 in eV/Å^3 internally)
p0 = [Es.min(), Vs[np.argmin(Es)], 15 / EV_A3_TO_GPA, 4.0]
pE, covE = curve_fit(bm3_E, Vs, Es, p0=p0, maxfev=20000)
E0, V0_E, B0_E, Bp_E = pE
errE = np.sqrt(np.diag(covE))
# pressure fit (GPa)
pP, covP = curve_fit(bm3_P, Vs, Ps, p0=[V0_E, B0_E * EV_A3_TO_GPA, Bp_E], maxfev=20000)
V0_P, B0_P, Bp_P = pP
errP = np.sqrt(np.diag(covP))

rmsE = 1000 * np.sqrt(np.mean((Es - bm3_E(Vs, *pE)) ** 2)) / len(fw_relaxed)
d2E = np.diff(Es, 2)
convex = bool(np.all(d2E > 0))
a_conv = lambda V: (4 * V) ** (1 / 3)        # FCC primitive (Z = 1 formula unit) -> conventional cubic a

print(f"E(V) BM3 : V0 = {V0_E:.2f} ± {errE[1]:.2f} Å^3 (a = {a_conv(V0_E):.4f} Å), "
      f"B0 = {B0_E*EV_A3_TO_GPA:.2f} ± {errE[2]*EV_A3_TO_GPA:.2f} GPa, B0' = {Bp_E:.2f} ± {errE[3]:.2f}, "
      f"fit RMS = {rmsE:.4f} meV/atom")
print(f"P(V) BM3 : V0 = {V0_P:.2f} ± {errP[0]:.2f} Å^3 (a = {a_conv(V0_P):.4f} Å), "
      f"B0 = {B0_P:.2f} ± {errP[1]:.2f} GPa, B0' = {Bp_P:.2f} ± {errP[2]:.2f}")
print(f"DFT training cell: V = {V_dft:.2f} Å^3 (a = {a_conv(V_dft):.4f} Å) -> "
      f"MACE V0 deviates by {100*(V0_E/V_dft-1):+.2f} % in volume")
print(f"|B0(E) − B0(P)| = {abs(B0_E*EV_A3_TO_GPA - B0_P):.2f} GPa | E(V) convex everywhere: {convex}")
SUMMARY.update({"T4_V0_A3": float(V0_E), "T4_a0_conv_A": float(a_conv(V0_E)),
                "T4_B0_energy_GPa": float(B0_E * EV_A3_TO_GPA), "T4_B0_pressure_GPa": float(B0_P),
                "T4_Bprime": float(Bp_E), "T4_V0_dev_from_DFT_cell_pct": float(100 * (V0_E / V_dft - 1)),
                "T4_convex": convex})

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SI-3 — Equation of state: E(V) with BM3 fit, and P(V) virial vs −dE/dV
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
Vf = np.linspace(Vs.min() * 0.995, Vs.max() * 1.005, 300)

ax = axes[0]
ax.plot(Vs, 1000 * (Es - E0) / len(fw_relaxed), "o", color="tab:blue", ms=4, label="MACE (relaxed ions)")
ax.plot(Vf, 1000 * (bm3_E(Vf, *pE) - E0) / len(fw_relaxed), "-", color="k", lw=1.0,
        label=f"BM3: B$_0$={B0_E*EV_A3_TO_GPA:.1f} GPa")
ax.axvline(V_dft, color="tab:red", ls="--", lw=0.8, label="DFT training cell")
ax.set_xlabel("V (Å$^3$ per primitive cell)", fontweight="bold"); ax.set_ylabel("E − E$_0$ (meV/atom)", fontweight="bold")
ax.legend(fontsize=7, frameon=False)
ax_top = ax.secondary_xaxis("top", functions=(a_conv, lambda a: a**3 / 4))
ax_top.set_xlabel("a (Å)", fontsize=9)
ax.text(-0.05, 1.16, "(a)", transform=ax.transAxes, fontsize=16, fontweight="bold", va="top")

ax = axes[1]
ax.plot(Vs, Ps, "s", color="tab:green", ms=4, label="MACE virial pressure")
ax.plot(Vf, bm3_P(Vf, *pP), "-", color="tab:green", lw=1.0, label=f"BM3 P(V): B$_0$={B0_P:.1f} GPa")
ax.plot(Vf, bm3_P(Vf, V0_E, B0_E * EV_A3_TO_GPA, Bp_E), "--", color="k", lw=0.9, label="$-dE/dV$ from (a)")
ax.axhline(0, color="0.5", lw=0.5); ax.axvline(V_dft, color="tab:red", ls="--", lw=0.8)
ax.set_xlabel("V (Å$^3$ per primitive cell)", fontweight="bold"); ax.set_ylabel("P (GPa)", fontweight="bold")
ax.legend(fontsize=7, frameon=False)
ax_top = ax.secondary_xaxis("top", functions=(a_conv, lambda a: a**3 / 4))
ax_top.set_xlabel("a (Å)", fontsize=9)
ax.text(-0.05, 1.16, "(b)", transform=ax.transAxes, fontsize=16, fontweight="bold", va="top")

fig.tight_layout()
savefig_dual(fig, "SI3_eos")
plt.show()
print("Saved SI3_eos.png/.pdf ->", FIG_DIR)


**Reading Test 4**
- *Convexity* and a smooth BM3 fit (fit RMS ≪ 1 meV/atom) show the model extrapolates physically over ±6 % volume.
- *B₀(E) ≈ B₀(P)* and overlap of the dashed and solid curves in (b) verify that MACE stresses are consistent with its energies.
- *V₀ vs DFT cell*: the DFT frames were computed at a fixed cell, so a MACE V₀ a few percent away is not by itself an error; it becomes informative if you know whether the DFT cell was PBE+D3-relaxed. Compare a₀ also with the experimental lattice parameter from your PXRD section, noting thermal expansion and the known tendencies of the functional.
- MACE gives B₀ ≈ 36.2–36.9 GPa (E(V) and P(V) fits, ±6% and ±2% windows all agree to ~2%; see Table 1) against the DFT reference B₀ = 41.0 GPa (Wu et al. 2013) — an ~10–12% underestimate. This is the same direction as the "too-soft" failure MOFSimBench reports for universal MLIPs on MOFs, though smaller in magnitude here; since this model never saw strained cells (no stress labels in training), adding a small set of ±2–4% strained DFT frames *with stress labels* is the standard remedy if tighter agreement is needed.

---
## Table 1 — Validation summary

Aggregates the headline numbers from Tests 1, 2, 3B and 4 into one table for the manuscript.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Table 1 — MACE UiO-66 (N₂ model) validation summary
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd
from scipy.optimize import curve_fit as _curve_fit

# ±2% volume window refit (vs. the ±6% / 13-point fit above), for B0 fit-range sensitivity
mask2 = np.abs(SCALES - 1.0) <= 0.021
pE2, _ = _curve_fit(bm3_E, Vs[mask2], Es[mask2], p0=[E0, V0_E, B0_E, Bp_E], maxfev=20000)
pP2, _ = _curve_fit(bm3_P, Vs[mask2], Ps[mask2], p0=[pE2[1], pE2[2] * EV_A3_TO_GPA, pE2[3]], maxfev=20000)
B0_E2, Bp_E2 = pE2[2] * EV_A3_TO_GPA, pE2[3]
B0_P2 = pP2[1]

r1x1x1 = results[SUPERCELLS[0]]
r2x2x2 = results[SUPERCELLS[-1]]
conv_cm1 = np.abs(np.sort(r1x1x1["fq"], axis=1) - np.sort(r2x2x2["fq"], axis=1)) * THZ_TO_CM1

# Wu, Yildirim & Zhou, J. Phys. Chem. Lett. 2013, 4, 925
DFT_B0, DFT_C11, DFT_C12, DFT_C44, DFT_a0 = 41.0, 59.35, 31.85, 17.63, 20.888

rows = [
    ("Energy accuracy (test)", "RMSE, N02 (118 atoms)",
     f"{table[('test','N02')]['E']['RMSE']:.3f} ({table[('test','N02')]['E']['RMSE']*118:.2f})", "—", "meV/atom (meV/cell)"),
    ("Energy accuracy (test)", "MAE, N02 (118 atoms)",
     f"{table[('test','N02')]['E']['MAE']:.3f} ({table[('test','N02')]['E']['MAE']*118:.2f})", "—", "meV/atom (meV/cell)"),
    ("Energy accuracy (test)", "RMSE, repulsion (118 atoms)",
     f"{table[('test','repulsion')]['E']['RMSE']:.3f} ({table[('test','repulsion')]['E']['RMSE']*118:.2f})", "—", "meV/atom (meV/cell)"),
    ("Energy accuracy (test)", "MAE, repulsion (118 atoms)",
     f"{table[('test','repulsion')]['E']['MAE']:.3f} ({table[('test','repulsion')]['E']['MAE']*118:.2f})", "—", "meV/atom (meV/cell)"),
    ("Energy accuracy (test)", "RMSE, gas N₂ (2 atoms)",
     f"{table[('test','gas_N2')]['E']['RMSE']:.3f} ({table[('test','gas_N2')]['E']['RMSE']*2:.2f})", "—", "meV/atom (meV/cell)"),
    ("Energy accuracy (test)", "MAE, gas N₂ (2 atoms)",
     f"{table[('test','gas_N2')]['E']['MAE']:.3f} ({table[('test','gas_N2')]['E']['MAE']*2:.2f})", "—", "meV/atom (meV/cell)"),
    ("Force accuracy (test)", "RMSE, all", f"{table[('test','all')]['F']['RMSE']:.2f}", "—", "meV/Å"),
    ("Force accuracy (test)", "MAE, all", f"{table[('test','all')]['F']['MAE']:.2f}", "—", "meV/Å"),
] + [
    ("Force accuracy (test)", f"RMSE, {el}", f"{table[('elem', el)]['test']['RMSE']:.2f}", "—", "meV/Å")
    for el in elements
] + [
    ("Phonons", "Lowest frequency off Γ, 1×1×1",
     f"{r1x1x1['min_path_off_gamma']:.3f} ({r1x1x1['min_path_off_gamma']*THZ_TO_CM1:.1f})", "—", "THz (cm$^{-1}$)"),
    ("Phonons", "Lowest frequency off Γ, 2×2×2",
     f"{r2x2x2['min_path_off_gamma']:.3f} ({r2x2x2['min_path_off_gamma']*THZ_TO_CM1:.1f})", "—", "THz (cm$^{-1}$)"),
    ("Phonons", "Acoustic Γ frequencies, 1×1×1", f"{np.abs(r1x1x1['gamma'][:3]).max():.1e} ($\\approx$0)", "0 (exact)", "THz"),
    ("Phonons", "Acoustic Γ frequencies, 2×2×2", f"{np.abs(r2x2x2['gamma'][:3]).max():.1e} ($\\approx$0)", "0 (exact)", "THz"),
    ("Phonons", "1×1×1 vs 2×2×2, mean $|\\Delta\\omega|$", f"{conv_cm1.mean():.2f}", "—", "cm$^{-1}$"),
    ("Phonons", "1×1×1 vs 2×2×2, max $|\\Delta\\omega|$", f"{conv_cm1.max():.2f}", "—", "cm$^{-1}$"),
    ("Mechanical properties", "$B_0$, $E(V)$ fit ($\\pm$6\\% window)", f"{B0_E*EV_A3_TO_GPA:.2f}", f"{DFT_B0:.1f}", "GPa"),
    ("Mechanical properties", "$B_0$, $E(V)$ fit ($\\pm$2\\% window)", f"{B0_E2:.2f}", f"{DFT_B0:.1f}", "GPa"),
    ("Mechanical properties", "$B_0$, $P(V)$ fit ($\\pm$6\\% window)", f"{B0_P:.2f}", f"{DFT_B0:.1f}", "GPa"),
    ("Mechanical properties", "$B_0$, $P(V)$ fit ($\\pm$2\\% window)", f"{B0_P2:.2f}", f"{DFT_B0:.1f}", "GPa"),
    ("Mechanical properties", "$B_0'$, $E(V)$ fit ($\\pm$6\\% / $\\pm$2\\%)", f"{Bp_E:.2f} / {Bp_E2:.2f}", "—", "—"),
    ("Mechanical properties", "$a_0=(4V_0)^{1/3}$, $E(V)$ fit", f"{a_conv(V0_E):.3f}", f"{DFT_a0:.3f}", "Å"),
    ("Mechanical properties", "MACE pressure at DFT cell volume", f"{P_dft_cell:.3f}", "0 (by definition)", "GPa"),
    ("Mechanical properties", "$C_{11}$", "— (not fit here)", f"{DFT_C11:.2f}", "GPa"),
    ("Mechanical properties", "$C_{12}$", "— (not fit here)", f"{DFT_C12:.2f}", "GPa"),
    ("Mechanical properties", "$C_{44}$", "— (not fit here)", f"{DFT_C44:.2f}", "GPa"),
]
if "1x1x1" in system_results and "2x2x2" in system_results:
    rows += [
        ("MD stability (LAMMPS NVE)", "Drift, 1×1×1 (118 atoms)",
         f"{system_results['1x1x1']['metrics']['drift_meV_atom_ps']*1000:.2e}", "—", "$\\mu$eV/atom/ps"),
        ("MD stability (LAMMPS NVE)", "Drift, 2×2×2 (944 atoms)",
         f"{system_results['2x2x2']['metrics']['drift_meV_atom_ps']*1000:.2e}", "—", "$\\mu$eV/atom/ps"),
    ]

table1_df = pd.DataFrame(rows, columns=["Section", "Quantity", "MACE", "Reference (DFT/exp)", "Unit"])
table1_df.to_csv(os.path.join(FIG_DIR, "Table1_MACE_UiO66_validation.csv"), index=False)

tex_lines = [
    "\\begin{table}[t]", "\\centering",
    "\\caption{MACE UiO-66 (N$_2$ model) validation summary. Reference $B_0$ and "
    "$C_{11}/C_{12}/C_{44}$: Wu, Yildirim \\& Zhou, \\emph{J. Phys. Chem. Lett.} "
    "\\textbf{2013}, 4, 925; DFT training-cell lattice parameter $a=20.888$ \\r{A}.}",
    "\\label{tab:mace_uio66_validation}",
    "\\begin{tabular}{llrrl}", "\\toprule",
    "Section & Quantity & MACE & Reference (DFT/exp) & Unit \\\\", "\\midrule",
]
prev = None
for _, r in table1_df.iterrows():
    sec = r["Section"] if r["Section"] != prev else ""
    if r["Section"] != prev and prev is not None:
        tex_lines.append("\\addlinespace")
    prev = r["Section"]
    tex_lines.append(f"{sec} & {r['Quantity']} & {r['MACE']} & {r['Reference (DFT/exp)']} & {r['Unit']} \\\\")
tex_lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
with open(os.path.join(FIG_DIR, "Table1_MACE_UiO66_validation.tex"), "w") as f:
    f.write("\n".join(tex_lines) + "\n")

print(f"Saved Table1_MACE_UiO66_validation.csv/.tex -> {FIG_DIR}")
table1_df


---
## Summary of reliability metrics

In [ ]:
def fmt(v):
    if isinstance(v, float): return f"{v:.4g}"
    if isinstance(v, (list, tuple)): return "[" + ", ".join(fmt(x) for x in v) + "]"
    return str(v)
w = max(len(k) for k in SUMMARY)
for k, v in SUMMARY.items():
    print(f"{k:{w}s} : {fmt(v)}")
with open(os.path.join(OUT_DIR, "validation_summary.json"), "w") as f:
    json.dump({k: (v if not isinstance(v, np.generic) else v.item()) for k, v in SUMMARY.items()}, f, indent=2,
              default=float)
print("\nsaved:", os.path.join(OUT_DIR, "validation_summary.json"))

### What these tests do and do not establish

| Test | Establishes | Does **not** establish |
|------|-------------|------------------------|
| 1. Held-out accuracy | Fit quality and absence of over-fitting on configurations like the training data | Transferability (random split of correlated MD frames = interpolation) |
| 2. Phonons | Correct local curvature, dynamical stability of the MACE minimum, harmonic C_v | Agreement with DFT phonons (needs a DFT reference run); LO–TO splitting; IR intensities |
| 3. NVE conservation | Smooth, conservative PES along 300 K trajectories of the production system | Correct dynamics/diffusivity by itself (that requires comparison with DFT-MD or experiment) |
| 3B. LAMMPS NVE, 1×1×1 vs 2×2×2 | That Test 3's conclusion holds on the actual production engine/model and does not degrade as the supercell grows | Full production-size convergence (only two sizes checked); still needs the same DFT-MD/experiment comparison as Test 3 |
| 4. EOS / bulk modulus | Physical extrapolation in volume; energy–stress consistency | Accuracy of the absolute B₀ without a same-level DFT reference |

**Recommended additions if a reviewer asks for more:** (i) a DFT phonon calculation of the same symmetrized structure using the saved Phonopy displacements; (ii) a committee of 3–5 MACE models trained with different `--seed` values — `MACECalculator(model_paths=[...])` then returns an energy/force variance usable as a per-configuration uncertainty estimate along MD trajectories; (iii) DFT single points on ~10 MACE-MD snapshots at 300 K that were *not* drawn from the DFT-MD trajectory, which is the cleanest out-of-distribution force test.

### References
- MOFSimBench: evaluating universal machine learning interatomic potentials in metal–organic framework molecular modeling, *npj Comput. Mater.* (2025). https://www.nature.com/articles/s41524-025-01872-3
- Machine learned potential for high-throughput phonon calculations of metal–organic frameworks (MACE-MP-MOF0), *npj Comput. Mater.* (2025). https://www.nature.com/articles/s41524-025-01611-8
- Wu, H.; Yildirim, T.; Zhou, W. Exceptional Mechanical Stability of Highly Porous Zirconium Metal–Organic Framework UiO-66 and Its Important Implications. *J. Phys. Chem. Lett.* 2013, 4, 925–930.
- Togo, A. First-principles phonon calculations with phonopy and phono3py. *J. Phys. Soc. Jpn.* 2023, 92, 012001.
- Batatia, I. et al. MACE: Higher order equivariant message passing neural networks for fast and accurate force fields. *NeurIPS* 2022.